# Bayesian Statistics

**Goal:** Work through a concrete Bayes-rule numeric example, implement Beta-Bernoulli conjugate updating, plot the prior-to-posterior shift as data accumulates, and contrast MAP vs MLE.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Bayes' Rule: Disease Test Example

A disease affects 1% of the population. A test has:
- **Sensitivity** (true positive rate): 99%
- **Specificity** (true negative rate): 95%

**Question:** If you test positive, what is the probability you have the disease?

```
P(D=1|T=+) = P(T=+|D=1) * P(D=1) / P(T=+)
```

In [2]:
# All quantities as plain Python floats for clarity
P_D = 0.01        # prior: P(disease)
P_TP = 0.99       # sensitivity: P(test+ | disease)
P_FP = 0.05       # false positive rate: P(test+ | no disease)

# P(test+) by total probability
P_T_pos = P_TP * P_D + P_FP * (1 - P_D)

# Posterior via Bayes' rule
P_D_given_T_pos = (P_TP * P_D) / P_T_pos

print(f"P(T=+)          = {P_T_pos:.4f}")
print(f"P(D=1 | T=+)    = {P_D_given_T_pos:.4f}  ({P_D_given_T_pos*100:.2f}%)")
print()
print("Despite a 99% sensitive test, a positive result only implies")
print(f"~{P_D_given_T_pos*100:.0f}% probability of disease — because the disease is rare (base rate 1%).")

P(T=+)          = 0.0594
P(D=1 | T=+)    = 0.1667  (16.67%)

Despite a 99% sensitive test, a positive result only implies
~17% probability of disease — because the disease is rare (base rate 1%).


## Beta-Bernoulli Conjugate Updating

We model a biased coin with unknown probability p. Our prior belief is `p ~ Beta(α, β)`.

After observing s heads and f tails:
```
p | data ~ Beta(α + s, β + f)
```

The posterior mean is `(α + s) / (α + β + s + f)`, which interpolates between the prior mean `α/(α+β)` and the MLE `s/(s+f)`.

In [3]:
# MPS does not support Beta distribution; compute on CPU
# (torch.distributions.Beta uses incomplete beta which isn't supported on MPS)
cpu = torch.device("cpu")

torch.manual_seed(42)
TRUE_P = 0.7  # true coin bias we want to recover
ALPHA0, BETA0 = 2.0, 2.0  # weakly informative prior (prior mean = 0.5)

# Generate a sequence of coin flips
N_FLIPS = 100
flips = torch.bernoulli(torch.full((N_FLIPS,), TRUE_P, device=cpu))

# p-grid for plotting
p_grid = torch.linspace(1e-4, 1 - 1e-4, 500, device=cpu)


def beta_log_pdf(p: torch.Tensor, alpha: float, beta: float) -> torch.Tensor:
    """Log-pdf of Beta(alpha, beta) evaluated at p."""
    dist = torch.distributions.Beta(
        torch.tensor(alpha, device=cpu), torch.tensor(beta, device=cpu)
    )
    return dist.log_prob(p)


def posterior_mean(alpha: float, beta: float) -> float:
    return alpha / (alpha + beta)


# Checkpoints at which to snapshot the posterior
checkpoints = [0, 5, 10, 30, 100]
fig, axes = plt.subplots(1, len(checkpoints), figsize=(15, 3), sharey=False)

alpha, beta = ALPHA0, BETA0
for i_cp, n_obs in enumerate(checkpoints):
    # Update alpha, beta with observations from previous checkpoint
    if i_cp > 0:
        prev = checkpoints[i_cp - 1]
        batch = flips[prev:n_obs]
        alpha += batch.sum().item()
        beta += (1 - batch).sum().item()

    pdf_vals = beta_log_pdf(p_grid, alpha, beta).exp()
    ax = axes[i_cp]
    ax.plot(p_grid.numpy(), pdf_vals.numpy())
    ax.axvline(TRUE_P, color="red", ls="--", lw=1, label="true p")
    ax.axvline(posterior_mean(alpha, beta), color="orange", ls=":", lw=1.5, label="post. mean")
    ax.set_title(f"n={n_obs}")
    ax.set_xlabel("p")
    if i_cp == 0:
        ax.set_ylabel("density")
        ax.legend(fontsize=7)

fig.suptitle(f"Beta-Bernoulli posterior update (true p={TRUE_P})", y=1.02)
plt.tight_layout()
plt.show()

print(f"Final posterior: Beta({alpha:.1f}, {beta:.1f})")
print(f"Posterior mean:  {posterior_mean(alpha, beta):.4f}  (true: {TRUE_P})")
print(f"MLE (heads/n):   {flips.mean().item():.4f}")

Final posterior: Beta(69.0, 35.0)
Posterior mean:  0.6635  (true: 0.7)
MLE (heads/n):   0.6700


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_12080/2174952382.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Assert posterior mean is within 0.1 of true p after 100 flips
final_post_mean = posterior_mean(alpha, beta)
empirical_rate = flips.mean().item()

assert abs(final_post_mean - TRUE_P) < 0.1, (
    f"Posterior mean {final_post_mean:.4f} too far from true p {TRUE_P}"
)
# With weak prior (alpha0=beta0=2) and 100 data points, posterior mean ≈ MLE
assert abs(final_post_mean - empirical_rate) < 0.05, (
    f"Posterior mean {final_post_mean:.4f} should be close to MLE {empirical_rate:.4f} with n=100"
)
print(f"Posterior mean ({final_post_mean:.4f}) is close to MLE ({empirical_rate:.4f}) with n=100 ✓")

Posterior mean (0.6635) is close to MLE (0.6700) with n=100 ✓


## MAP vs MLE

**MLE** maximises the likelihood `p(data | θ)` — ignores the prior.

**MAP** maximises the posterior `p(θ | data) ∝ p(data | θ) · p(θ)` — equivalent to MLE + log-prior regularisation.

For Beta-Bernoulli, the MAP estimate is the mode of Beta(α, β):

```
θ_MAP = (α - 1) / (α + β - 2)   for α, β > 1
θ_MLE = s / (s + f)
```

In [5]:
# Small-sample regime: 3 heads out of 5 flips
torch.manual_seed(0)
small_flips = torch.tensor([1.0, 1.0, 1.0, 0.0, 0.0], device=cpu)  # 3 heads, 2 tails

alpha_small = ALPHA0 + small_flips.sum().item()   # 2 + 3 = 5
beta_small  = BETA0 + (1 - small_flips).sum().item()  # 2 + 2 = 4

mle_small  = small_flips.mean().item()                           # 3/5 = 0.6
map_small  = (alpha_small - 1) / (alpha_small + beta_small - 2)  # (5-1)/(9-2) = 4/7
post_mean_small = posterior_mean(alpha_small, beta_small)         # 5/9

print(f"n=5 flips (3 heads):")
print(f"  MLE        = {mle_small:.4f}  (s/n = 3/5)")
print(f"  MAP        = {map_small:.4f}  (mode of Beta(5,4))")
print(f"  Post. mean = {post_mean_small:.4f}  (mean of Beta(5,4))")

# With more data, all three converge
alpha_big = ALPHA0 + alpha - ALPHA0  # use final alpha from 100 flips
beta_big = BETA0 + beta - BETA0
mle_big = (alpha_big - ALPHA0) / ((alpha_big - ALPHA0) + (beta_big - BETA0))
map_big = (alpha_big - 1) / (alpha_big + beta_big - 2)
post_mean_big = posterior_mean(alpha_big, beta_big)

print(f"\nn=100 flips:")
print(f"  MLE        = {mle_big:.4f}")
print(f"  MAP        = {map_big:.4f}")
print(f"  Post. mean = {post_mean_big:.4f}")
print("As n → ∞, MAP ≈ MLE ≈ posterior mean (prior gets swamped by data).")

n=5 flips (3 heads):
  MLE        = 0.6000  (s/n = 3/5)
  MAP        = 0.5714  (mode of Beta(5,4))
  Post. mean = 0.5556  (mean of Beta(5,4))

n=100 flips:
  MLE        = 0.6700
  MAP        = 0.6667
  Post. mean = 0.6635
As n → ∞, MAP ≈ MLE ≈ posterior mean (prior gets swamped by data).


## Takeaways

- **Bayes' rule in practice:** even a 99%-sensitive test can have low positive predictive value when the prior (base rate) is very low. This is the base-rate fallacy.
- **Beta-Bernoulli conjugacy:** the posterior stays in the Beta family; the update is trivially `α → α+s, β → β+f`. The posterior mean interpolates prior belief and empirical rate.
- **MAP vs MLE:** MAP is MLE with a regularisation term. With small data, MAP is pulled toward the prior; with lots of data, MAP ≈ MLE.
- **AI engineering connection:** L2 regularisation on weights = Gaussian prior on parameters; L1 regularisation = Laplace prior. Thompson sampling uses the posterior for exploration.